# పాఠము 04 - సాధనం ఉపయో​గ డిజైన్ నమూనా

ఈ పాఠంలో మీరు Microsoft Agent Framework (Python) ఉపయోగించి AI ఏజెంట్లు కోసం **సాధనం ఉపయో​గ** డిజైన్ నమూనాను నేర్చుకుంటారు. మేము కవర్ చేస్తున్న అంశాలు:

- `@tool` డెకొరేటర్ మరియు టైప్ చేయబడిన పారామితులతో ఫంక్షన్ సాధనాలను నిర్వచించడం
- మోడల్ ప్రతి సాధనం ఏమి చేస్తుందో తెలిసేందుకు సాధన స్కీమాలను అందించడం
- `approval_mode` తో సాధన అమలును నియంత్రించడం
- Pydantic నమూనాలు మరియు `response_format` ద్వారా **సంఘటిత అవుట్పుట్** ను ఇవ్వడం

సన్నివేశం ఒక **ప్రయాణ బుకింగ్ ఏజెంట్** కావడం, ఇది గమ్యస్థలాలను చూడగలదు, లభ్యతను తనిఖీ చేయగలదు మరియు విమాన సమాచారం పొందగలదు.


## సెట్ అప్


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool డెకోరేటర్‌తో పరికరాలను నిర్వచించడం

`@tool` డెకోరేటర్ సాధారణ Python ఫంక్షన్‌ని ఏజెంట్ కాల్ చేయగల పరికరంగా మారుస్తుంది.  
ప్రధాన విషయాలు:

- **డాక్స్ట్రింగ్** మోడల్ చూసే పరికర వివరణగా మారుతుంది.  
- **టైప్ అనోటేషన్లు** (`Annotated` సహా వివరణలతో) పరికర స్కీమా నిర్వచిస్తాయి.  
- `approval_mode` యూజర్ ప్రతి కాల్ అమలు కాకముందు ఆమోదించాల్సిన అవసరముంటుందో లేదో నియంత్రిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## బహుళ టూల్స్‌తో ఏజెంట్ సృష్టించడం

మోడల్ వినియోగదారుడి ప్రశ్నకు సమాధానం చెప్పేందుకు అవసరమైన ఏమయితే వాటినైనా పిలిచేందుకు అన్ని మూడు టూల్స్‌ను క్లయింట్‌కు పంపండి.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## టూల్స్‌తో సరంజామా ఉత్పత్తి

`response_format` ను Pydantic మోడల్‌కు సెట్ చేయడం వలన, ఏజెంట్ ఆజ్ఞాత పాఠ్యాన్ని కాకుండా మంచి టైప్ చేయబడిన JSON αντικ-object ను తిరిగి ఇవ్వాల్సి ఉంటుంది. ఇది దిగువ కోడ్ ఫలితాన్ని ప్రోగ్రామాటిక్గా ఉపయోనించుకోవాలనుకున్నప్పుడు ఉపయోగపడుతుంది.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## టూల్ ఆమోద నమూనాలు

`@tool`లోని `approval_mode` పారామీటర్ టూల్ కాల్స్ అమలుచేయక ముందు మనిషి ఆమోదం అవసరమా లేదో నియంత్రిస్తుంది:

| మోడ్ | ప్రవర్తన |
|---|---|
| `"never_require"` | టూల్ స్వయంచాలకంగా పరుగెడుతుంది — యూజర్ నిర్ధారణ అవసరం లేదు. |
| `"always_require"` | ప్రతి కాల్ అమలుచేయక ముందు యూజర్ ఆమోదం పొందాలి. |

పక్కప్రభావాలు ఉండే టూల్స్ (ఉదా: జెట్ బుక్ చేయడం, క్రెడిట్ కార్డు ఛార్జ్ చేయడం) కోసం `"always_require"` ని ఉపయోగించండి, తద్వారా మనిషి వ్యవస్థలో ఉంటుంది.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

ఈ పాఠంలో మీరు తెలుసుకున్నది:

1. **@tool** డెకొరేటర్ ఉపయోగించి టైప్డ్ పారామితులు మరియు టూల్ స్కీమ్ గా పనిచేసే డోక్స్‌ట్రింగ్స్ తో **టూల్స్** ని నిర్వచించడం.
2. **ఎన్నికా ప్రశ్నలకు స్పందించడానికి ఏజెంట్ వాటిని వరుసగా పిలవడానికి అనేక టూల్స్** కలపడం.
3. **Pydantic మోడల్‌ను `response_format` గా పాస్ చేసి గঠনాత్మక అవుట్‌పుట్**ను రిటర్న్ చేయడం.
4. సున్నితమైన కార్యకలాపాల కోసం మానవ నియంత్రణతో **`approval_mode`తో టూల్ ఆమోదాన్ని నియంత్రించడం**.

ఈ నమూనాలు నమ్మక లాంటి, ఉత్పత్తి-తయారైన ఏజెంట్లను నిర్మించడానికి, అవి బాహ్య వ్యవస్థలతో సురక్షితంగా పరస్పరం చర్య చేయగలవు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్పష్టత**:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించడం జరిగింది. మేము సరిగా ఉండేందుకు ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలలో పొరపాట్లు లేదా లోపభూయిష్టతలు ఉండవచ్చు. ఆ పత్రం స్వదేశీ భాషలో ఉన్నది అధికారిక మూలంగా పరిగణించాలి. ముఖ్యమైన సమాచారానికి, నిపుణుల చేత మానవ అనువాదాన్ని సూచిస్తాము. ఈ అనువాదం వలన జరిగే ఏవైనా తప్పుదోవలు లేదా గందరగోళాలకు మేము బాధ్యత లేకపోతున్నాము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
